In [ ]:
import pyro as py
from data_retrieval.database_io import DB_handler
import pandas as pd
import numpy as np

In [ ]:
dbh = DB_handler()

In [ ]:
start = dbh.valuations.analysis_valuations(0)

In [ ]:
half_year = dbh.valuations.analysis_valuations(52)

In [ ]:
half_year

In [ ]:
half_year = np.array(half_year)
half_year = pd.DataFrame(half_year, columns=["id", "krz", "broker", "target_usd", "price"])
half_year = half_year.astype({"id": "int", "krz": "string", "broker": "string", "target_usd": "float","price": "float"})

In [ ]:
start = np.array(start)
start = pd.DataFrame(start, columns=["id", "krz", "broker", "target_usd", "price"])
start = start.astype({"id": "int", "krz": "string", "broker": "string", "target_usd": "float","price": "float"})

In [ ]:
merged = start.merge(half_year, how="inner", on="id", suffixes=("", "_half_year"))

In [ ]:
merged["target_p"] = (merged.target_usd/merged.price)-1
merged["half_year_p"] = (merged.price_half_year/merged.price)-1

In [ ]:
merged["target_y"] = merged.half_year_p - merged.target_p

In [ ]:
# only positive targets
merged = merged[merged["target_p"] > 0]

In [ ]:
merged

In [ ]:
barclays = merged[merged["broker"] == "Barclays"]
barclays["color"] = np.where(barclays["target_p"] < 0.1, "<0.1", np.where(barclays["target_p"] < 0.25, "<0.25", ">0.25"))

In [ ]:
barclays.sort_values("target_p", ascending=False)

In [ ]:
import plotly.express as px

px.histogram(barclays[barclays["target_p"] < 1], x="target_y")

In [ ]:
# group merged by broker and count, order by most

merged.groupby("broker").count().sort_values("id", ascending=False)

In [ ]:
merged_ = merged.copy()
merged = merged[merged["broker"].isin(["Barclays", "Morgan Stanley", "JPMorgan Chase & Co.", "Citigroup", "Royal Bank of Canada"])] #

In [ ]:
merged.broker.unique()

In [ ]:
merged = merged[merged["target_p"] <= 1]

In [ ]:
import torch


In [ ]:
import torch
import pyro as pyro
import pyro.distributions as dist
from pyro.infer import MCMC, NUTS
import matplotlib.pyplot as plt
import seaborn as sns

# Number of independent normal distributions
X = len(merged.broker.unique())

# Step 1: Define the Bayesian model
def model(data_list):
    with pyro.plate("normals", X):
        mu = pyro.sample("mu", dist.Normal(0, 1))  # Prior for each mean
        sigma = pyro.sample("sigma", dist.HalfNormal(1))  # Prior for each std dev
    
    for i in range(X):  # Iterate over different distributions
        pyro.sample(f"obs_{i}", dist.Normal(mu[i], sigma[i]), obs=data_list[i])

# Step 2: Data
data_list = [torch.tensor(merged[merged["broker"] == b].target_y.values, dtype=torch.float) for b in merged.broker.unique()]

# Step 3: Run inference using NUTS
nuts_kernel = NUTS(model)
mcmc = MCMC(nuts_kernel, num_samples=1000, warmup_steps=200)
mcmc.run(data_list)

# Step 4: Get posterior samples
samples = mcmc.get_samples()
mu_samples = samples["mu"].numpy()  # Shape (1000, X)
sigma_samples = samples["sigma"].numpy()  # Shape (1000, X)

# Step 5: Visualization
def plot_posterior(samples, param_name):
    """Plots posterior histograms and trace plots for all X distributions"""
    X = samples.shape[1]
    fig, axs = plt.subplots(X, 2, figsize=(12, 4 * X))

    for i in range(X):
        # Trace plot
        axs[i, 0].plot(samples[:, i], alpha=0.7)
        axs[i, 0].set_title(f"Trace Plot of {param_name}_{i}")
        axs[i, 0].set_xlabel("Sample Index")
        axs[i, 0].set_ylabel(f"{param_name}_{i}")

        # Posterior distribution
        sns.histplot(samples[:, i], bins=30, kde=True, ax=axs[i, 1])
        # axs[i, 1].axvline(true_values[i], color="red", linestyle="dashed", label="True Value")
        axs[i, 1].legend()
        axs[i, 1].set_title(f"Posterior Distribution of {param_name}_{i}")
        axs[i, 1].set_xlabel(f"{param_name}_{i}")

    plt.tight_layout()
    plt.show()

# Plot mu and sigma
plot_posterior(mu_samples, "mu")
plot_posterior(sigma_samples, "sigma")


In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns

# Function to make predictions using posterior samples
def predict(mcmc_samples, num_predictions=100, X=3):
    mu_samples = mcmc_samples["mu"]  # Shape (num_samples, X)
    sigma_samples = mcmc_samples["sigma"]  # Shape (num_samples, X)
    
    predictions = {}
    
    for i in range(X):
        # Sample from the posterior predictive distribution
        pred_samples = torch.normal(mu_samples[:, i], sigma_samples[:, i])  # Shape (num_samples,)
        
        # Store results
        predictions[f"dist_{i}"] = pred_samples.numpy()
        
        # Plot the predictive distribution
        plt.figure(figsize=(6, 4))
        sns.histplot(pred_samples.numpy(), bins=30, kde=True)
        plt.axvline(mu_samples[:, i].mean(), color='red', linestyle='dashed', label="Mean Prediction")
        print(mu_samples[:, i].mean())
        print(mu_samples[:, i].std())
        plt.title(f"Predictive Distribution for Distribution {i}")
        plt.xlabel("Predicted Value")
        plt.ylabel("Density")
        plt.legend()
        plt.show()
    
    return predictions

# Step 1: Get posterior samples
samples = mcmc.get_samples()

# Step 2: Generate predictions
predictions = predict(samples, num_predictions=1000, X=5)
